In [1]:

import pandas as pd
import numpy as np

# =====================================
# 1. LOAD DATASET
# =====================================
df = pd.read_csv("factory-batch-measurements.csv")

print("Original Dataset:")
print(df.head())

print("\nDataset Shape:", df.shape)

# Select numeric columns
num = df.select_dtypes(include="number")

# =====================================
# 2. IQR METHOD
# =====================================
Q1 = num.quantile(0.25)
Q3 = num.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

iqr_summary = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "Lower Bound": lower,
    "Upper Bound": upper
})

print("\nIQR Summary:")
print(iqr_summary.round(3))

# Detect IQR outliers
iqr_outliers = (num.lt(lower) | num.gt(upper))

print("\nIQR Outlier Count per Column:")
print(iqr_outliers.sum())

print("\nTotal IQR Outlier Cells:", iqr_outliers.sum().sum())

# =====================================
# 3. Z-SCORE METHOD
# =====================================
mean = num.mean()
std = num.std(ddof=0)

z_scores = (num - mean) / std.replace(0, np.nan)

# Outlier threshold: absolute Z-score > 3
z_outliers = z_scores.abs() > 3

print("\nZ-Score Outlier Count per Column:")
print(z_outliers.sum())

print("\nTotal Z-Score Outlier Cells:", z_outliers.sum().sum())

# Display Z-score outlier records
for col in num.columns:
    mask = z_outliers[col]

    print(f"\nZ-Score Outliers in {col}:")

    if mask.any():
        print(df.loc[mask, ["BatchID", col]].assign(
            Z_Score=z_scores.loc[mask, col].round(3)
        ))
    else:
        print("No outliers found")

# =====================================
# 4. IQR CAPPING
# =====================================
df_capped = df.copy()

for col in num.columns:
    df_capped[col] = df_capped[col].clip(
        lower=lower[col],
        upper=upper[col]
    )

# Count capped values per column
capped_count = (
    df[num.columns] != df_capped[num.columns]
).sum()

print("\nCapped Values per Column:")
print(capped_count)

print("\nTotal Values Capped:", capped_count.sum())

# =====================================
# 5. COMPARE ORIGINAL AND CAPPED DATA
# =====================================
print("\nOriginal Numeric Data:")
print(df[num.columns].head())

print("\nCapped Numeric Data:")
print(df_capped[num.columns].head())

# =====================================
# 6. VERIFY CAPPING
# =====================================
remaining_outliers = (
    df_capped[num.columns].lt(lower) |
    df_capped[num.columns].gt(upper)
)

print("\nRemaining IQR Outliers After Capping:")
print(remaining_outliers.sum())

# =====================================
# 7. SAVE CAPPED DATASET
# =====================================
df_capped.to_csv("factory-batch-measurements-capped.csv",
                 index=False)

print("\nCapped dataset saved successfully!")


Original Dataset:
  BatchID ProductionDate Shift  Temperature_C  Pressure_PSI  Humidity_pct  \
0   B-001     2025-01-03     A           68.5         112.4          41.2   
1   B-002     2025-01-03     B           69.1         111.8          42.5   
2   B-003     2025-01-04     C           71.3         115.6          39.8   
3   B-004     2025-01-04     A           70.4         113.2          40.1   
4   B-005     2025-01-05     B           72.0         114.7          38.9   

   DefectCount  PassRate_pct                                       Notes  
0            2          98.0                    Baseline calibration run  
1            1          99.2            Normal variance within tolerance  
2            4          94.5  Slight uptick in defects after filter swap  
3            0         100.0                         All sensors nominal  
4            3          96.8              Operator noted brief vibration  

Dataset Shape: (60, 9)

IQR Summary:
                    Q1       Q3